In [1]:
import jax
import jax.numpy as jnp
from jax import lax, jit
from functools import partial
import time

# --- Configuration ---
MAX_RECURSION_DEPTH    = 1_000_000
OPTIMAL_DEPTH_STEP     = 250_000
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE            = 50_000_000  # e.g., 50M samples

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    """Normalizes depth scaling to prevent instability."""
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth", "scale_factor"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    """
    A single run of up to `depth` iterations, stabilized and with dynamic pi/phi.
    The loop body is compiled once for a given static depth.
    """
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    depth = stabilize_depth(depth)

    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        return new_val

    # We convert depth to int to use fori_loop
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)


@partial(jit, static_argnames=["total_depth", "scale_factor"])
def chunked_dppu(x, total_depth, scale_factor=1.0):
    """
    A single JIT-compiled function that handles multiple 250k-step chunks internally,
    rather than making repeated calls from Python. This reduces overhead.

    total_depth: total recursion steps you want (must be multiple of 250k in this example)
    scale_factor: scale for pi/phi computations
    """
    # Number of chunks of size OPTIMAL_DEPTH_STEP to run
    iterations = total_depth // OPTIMAL_DEPTH_STEP

    # The chunk body: each chunk runs dppu_with_dynamic_pi_phi for 250k iterations
    def chunk_body(_, val):
        return dppu_with_dynamic_pi_phi(val, depth=OPTIMAL_DEPTH_STEP, scale_factor=scale_factor)

    # We do 'iterations' chunks in one single fori_loop
    return lax.fori_loop(0, iterations, chunk_body, x)


def process_with_larger_depths(x, total_depth):
    """
    High-level function for clarity.
    This calls the single JIT function (chunked_dppu) with the desired total_depth.
    """
    # For example, we fix scale_factor=0.5 for demonstration
    return chunked_dppu(x, total_depth, scale_factor=0.5)


# ----------------------------------------------------------------
# Example usage & timing
if __name__ == "__main__":
    # Prepare a large batch input
    batch_input = jnp.linspace(0, 10, BATCH_SIZE)

    # Warm-up: compile everything once at 250k
    _ = chunked_dppu(batch_input, 250_000, scale_factor=0.5)

    # Try different total_depth values
    for depth in [250_000, 500_000, 1_000_000]:
        start_time = time.time()
        output = process_with_larger_depths(batch_input, depth)
        # Force materialization on host to measure full time
        output_blocking = jax.device_get(output)
        end_time = time.time()

        print(f"Depth={depth} => Output shape: {output_blocking.shape}, time: {end_time - start_time:.4f} sec")


Depth=250000 => Output shape: (50000000,), time: 47.8552 sec
Depth=500000 => Output shape: (50000000,), time: 48.4711 sec
Depth=1000000 => Output shape: (50000000,), time: 96.2055 sec
